# Miniprot divergence threhold determination

Let's look at some of the log files between mmseqs and miniprot to see if we can infer good threholds model of the latter based on the former

In [5]:
from pathlib import Path
import re
import sys
from dataclasses import dataclass, field

from Bio import SeqIO
import polars as pl
from intervaltree import Interval, IntervalTree

In [6]:
data_dir = Path("/home/yjkbertrand/Documents/projects/nextpyper_simulation/miniprot_divergence/data/batrachium")
mmseqs_dir = data_dir / "mmseqs_logs"
miniprot_dir = data_dir / "miniprot_logs"

The good thing is that the mmseqs logs are eqasy to parse, they are structured already

In [7]:
pl.read_csv(mmseqs_dir / "BA_28_1.log", separator="\t")

query,theader,cis,nident,mismatch,gapopen,tlen,tprobe,adj_cov,cov,idt,eff_cov
str,str,bool,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""BA_28_1|BA_28_1-6152_EDGE_66_l…","""probe_6152""",true,380,17,0,572,6152,386,0.674825,0.957179,0.645928
"""BA_28_1|K22_14_1-5973_EDGE_14_…","""probe_5973""",false,205,7,0,426,5973,207,0.485915,0.966981,0.469871
"""BA_28_1|IT_07_1-6152_EDGE_1209…","""probe_6152""",true,103,6,1,572,6152,106,0.185315,0.944954,0.175114
"""BA_28_1|SP_04_4-4816_EDGE_733_…","""probe_4816""",true,91,3,0,595,4816,93,0.156303,0.968085,0.151314
"""BA_28_1|BA_28_1-672_EDGE_2051_…","""probe_672""",true,472,63,3,705,672,472,0.669504,0.882243,0.590665
…,…,…,…,…,…,…,…,…,…,…,…
"""BA_28_1|SW_13_1-7581_EDGE_1168…","""probe_7581""",true,309,36,0,623,7581,334,0.536116,0.895652,0.480173
"""BA_28_1|K22_16_1-9449_EDGE_258…","""probe_9449""",true,103,1,0,358,9449,102,0.284916,0.990385,0.282177
"""BA_28_1|BA_28_1-7338_EDGE_131_…","""probe_7338""",true,105,8,0,526,7338,104,0.197719,0.929204,0.183721


In [8]:
mmseqs_tables = {file.stem:pl.read_csv(file, separator="\t") for file in sorted(mmseqs_dir.glob("*.log"))}

In [9]:
len(mmseqs_tables)

232

In [10]:
mmseqs_tables["B16_001_01"]

query,theader,cis,nident,mismatch,gapopen,tlen,tprobe,adj_cov,cov,idt,eff_cov
str,str,bool,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""B16_001_01|K17_49_1-7680_EDGE_…","""probe_7680""",true,94,6,0,589,7680,99,0.168081,0.94,0.157997
"""B16_001_01|B16_001_01-6955_EDG…","""probe_6955""",true,362,9,0,460,6955,365,0.793478,0.975741,0.774229
"""B16_001_01|B16_001_01-9988_EDG…","""probe_9988""",true,259,17,1,521,9988,264,0.506718,0.938406,0.475507
"""B16_001_01|B16_001_01-9344_EDG…","""probe_9344""",true,436,29,2,450,9344,448,0.995556,0.937634,0.933467
"""B16_001_01|B16_001_01-9087_EDG…","""probe_9087""",true,99,2,0,609,9087,98,0.16092,0.980198,0.157733
…,…,…,…,…,…,…,…,…,…,…,…
"""B16_001_01|SW_13_1-2100_EDGE_7…","""probe_2100""",true,192,3,1,460,2100,191,0.415217,0.984615,0.408829
"""B16_001_01|K22_16_1-1568_EDGE_…","""probe_1568""",true,137,11,0,949,1568,141,0.148577,0.925676,0.137535
"""B16_001_01|IT_44_06-3393_EDGE_…","""probe_3393""",true,329,32,1,498,3393,337,0.676707,0.911357,0.616722


In [143]:
import plotly.express as px
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
df = mmseqs_tables["SP_23_3"].filter(pl.col('cov') > 0.8).select("idt")
nbins = len(np.histogram_bin_edges(df, bins='auto', range=(0.6, 1)))
print(nbins)
fig = px.histogram(df, x="idt", nbins=nbins)
fig.show()

f = fig.full_figure_for_development(warn=False)
xbins = f.data[0].xbins

plotbins = list(np.arange(start=xbins['start'], stop=xbins['end']+xbins['size'], step=xbins['size']))
counts, bins = np.histogram(df, bins=plotbins)
# print(counts, bins)
def gaussian(x, mu, sigma, a):
    return a*np.exp(-(x-mu)**2/2/sigma**2)
def bimodal(x,mu1,sigma1,A1,mu2,sigma2,A2):
    return gaussian(x,mu1,sigma1,A1)+gaussian(x,mu2,sigma2,A2)

gau_params, gau_covariance = curve_fit(gaussian, bins[:-1], counts)
bi_params, bi_covariance=curve_fit(bimodal, bins[:-1], counts)
a_fit, b_fit, c_fit = gau_params
print(f"Gaussian fit: {a_fit}, {b_fit}, {c_fit}")
a0_fit, b0_fit, c0_fit, a1_fit, b1_fit, c1_fit = bi_params
print(f"Bimodal fit: {a0_fit}, {b0_fit}, {c0_fit}, {a1_fit}, {b1_fit}, {c1_fit}")
# plt.scatter(bins[:-1], counts, label='Data')
# x_fine = np.linspace(xbins['start'], 1)
# plt.plot(x_fine, gaussian(x_fine, a_fit, b_fit, c_fit), 'r-', label='gaussian fit')
# plt.plot(x_fine, bimodal(x_fine, a0_fit, b0_fit, c0_fit, a1_fit, b1_fit, c1_fit), 'b-', label='bimodal fit')
# plt.legend()
# plt.show()
from sklearn.mixture import GaussianMixture
X = np.array([np.array([x, y]) for x, y in zip(bins[:-1], counts) if y>10])
gm_2 = GaussianMixture(n_components=2, random_state=0).fit(X )
print(f"bic 2 is {gm_2.bic(X)}")
print(f'mu1={gm_2.means_[0]},) mu2={gm_2.means_[1]}')
print(f'sigma1={np.sqrt(gm_2.covariances_[0])}, sigma2={np.sqrt(gm_2.covariances_[1])}')
print(f'w1={gm_2.weights_[0]}, w2={gm_2.weights_[1]}')
print(f'n1={int(nbins * gm_2.weights_[0])} n2={int(nbins * gm_2.weights_[1])}')
gm_1 = GaussianMixture(n_components=1, random_state=0).fit(X)
print(f"bic 1 is {gm_1.bic(X)}")
print(f'mu1={gm_1.means_[0]}')
print(f'sigma1={np.sqrt(gm_1.covariances_[0])}')
print(f'w1={gm_1.weights_[0]}')
print(f'n1={int(nbins * gm_1.weights_[0])}')
# plt.scatter(bins[:-1], counts, label='Data')
# logprob = gm.score_samples(list(zip(bins[:-1], counts)))
# pdf = np.exp(logprob)
# plt.plot(list(zip(bins[:-1], counts)), pdf, '-r')
# fig = px.scatter(df, x="idt", y='cov')



73


Gaussian fit: 0.9501051807399288, -0.025246654049604262, 207.84259783794442
Bimodal fit: 0.9501052325391931, 0.025246691912075424, 207.84244788729055, 1.604525984268753, 0.028021914380083696, 202.9505314120937
bic 2 is 187.66815023179524
mu1=[  0.94774582 198.03036223],) mu2=[ 0.93060945 78.61791197]
sigma1=[[1.04075126e-02 7.26251541e-01]
 [7.26251541e-01 5.47167842e+01]], sigma2=[[4.18672622e-02 8.37326534e-01]
 [8.37326534e-01 4.73765167e+01]]
w1=0.28792716300946763, w2=0.7120728369905324
n1=21 n2=51
bic 1 is 187.66027431620589
mu1=[  0.93554348 113.        ]
sigma1=[[3.66000455e-02 1.03472324e+00]
 [1.03472324e+00 7.33745602e+01]]
w1=1.0
n1=73


/tmp/ipykernel_4141677/442748542.py:23: OptimizeWarning:

Covariance of the parameters could not be estimated



In [146]:
#Make a grid search for the best combination of number of gaussian distributions and covariance type
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_blobs

#X, _ = make_blobs(n_samples=10_000, centers=3)

param_grid = {
    'n_components': range(1, 7),
    'covariance_type': ['spherical', 'tied', 'diag', 'full']
}

def bic_scorer(fitted_gmm, X):
    return -fitted_gmm.bic(X)  # return -bic as higher is better for GridSearchCV

gs = GridSearchCV(GaussianMixture(), param_grid=param_grid, scoring=bic_scorer)
gs.fit(X)
gs.best_score_, gs.best_params_

{'scoring': <function bic_scorer at 0x7cf4e6f863e0>, 'estimator': GaussianMixture(), 'n_jobs': None, 'refit': True, 'cv': None, 'verbose': 0, 'pre_dispatch': '2*n_jobs', 'error_score': nan, 'return_train_score': False, 'param_grid': {'n_components': range(1, 7)}, 'multimetric_': False, 'best_index_': np.int64(0), 'best_score_': np.float64(-67.46687901167499), 'best_params_': {'n_components': 1}, 'best_estimator_': GaussianMixture(), 'refit_time_': 0.0018267631530761719, 'scorer_': <function bic_scorer at 0x7cf4e6f863e0>, 'cv_results_': {'mean_fit_time': array([0.00221815, 0.00642219, 0.00601931, 0.0078752 , 0.00604362,
       0.00390067]), 'std_fit_time': array([0.00035986, 0.00289878, 0.00341829, 0.00663253, 0.0020367 ,
       0.00020901]), 'mean_score_time': array([0.00037489, 0.00040421, 0.000386  , 0.00037932, 0.00045538,
       0.00048985]), 'std_score_time': array([4.35234014e-05, 4.57432249e-05, 6.04043716e-05, 3.78808034e-05,
       7.84754309e-05, 6.84151823e-05]), 'param_n_co

(np.float64(-67.46687901167499), {'n_components': 1})

In [108]:
df = mmseqs_tables["SP_24_1"].filter(pl.col('cov') > 0.5)
fig = px.histogram(df, x="idt", nbins=50)
fig.show()

In [106]:
keys = ["SE_14_1","SE_17_1","SE_20_1","SE_24_1"]
dfs = [mmseqs_tables[key].filter(pl.col('cov') > 0.5).with_columns(pl.lit(key).alias("acc")) for key in keys ]
new_df = pl.concat(dfs)
#new_df
fig = px.histogram(new_df, x="idt", nbins=500,color="acc")
fig.show()

In [45]:
df = mmseqs_tables["SP_24_1"]
fig = px.scatter(df, x="idt", y='cov')
fig.show()

In [148]:
mmseqs_dir = Path("/home/yjkbertrand/Documents/projects/nextpyper_simulation/miniprot_divergence/scfs_filtering")
mmseqs_tables_hier = {file.stem:pl.read_csv(file, separator="\t") for file in sorted(mmseqs_dir.glob("*.log"))}

In [172]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
dfs = [mmseqs_tables_hier[key].filter(pl.col('cov') > 0.1).with_columns(pl.lit(key).alias("acc")).select("idt") for key in mmseqs_tables_hier.keys() ]
total_rows = len(dfs)
total_cols = 1
fig = make_subplots(rows=total_rows, cols=total_cols, shared_xaxes=True, vertical_spacing=0.02,subplot_titles=list(mmseqs_tables_hier.keys()),
                    shared_yaxes=True)

fig.update_layout(height=2000, width=1500, title_text=f"Similarity histograms",
    template="plotly")
for i, df in enumerate(dfs):
    fig.add_trace(go.Histogram(x=df["idt"],nbinsx=100),row=i+1,col=1)
    #fig.add_trace(go.Scatter(x=df["idt"],y=df['cov']))
# new_df = pl.concat(dfs)
# #new_df
# fig = px.histogram(new_df, x="idt", nbins=500,color="acc")
fig.show()

In [171]:
from polars import DataFrame
def get_histogram_data(df:DataFrame, min_range:float=0.5) ->tuple[list, list]:
    """"
    Automatically infer the best number of bins in the histogram.
    Return x, y coordinates with y the height of the bar and y the central value of the bar.
    """
    nbins = len(np.histogram_bin_edges(df, bins='auto', range=(min_range, 1)))
    tmp_fig = px.histogram(df, x="idt", nbins=nbins)
    f = tmp_fig.full_figure_for_development(warn=False)
    xbins = f.data[0].xbins
    plotbins = list(np.arange(start=xbins['start'], stop=xbins['end']+xbins['size'], step=xbins['size']))
    counts, bins = np.histogram(df, bins=plotbins)
    return (bins, counts)
param_grid = {
    'n_components': range(1, 7),
    'covariance_type': ['spherical', 'tied', 'diag', 'full']
}
histogram_data = [get_histogram_data(df) for df in dfs]
Xs = [np.array([np.array([x, y]) for x, y in zip(item[0][:-1], item[1]) if y>10]) for item in histogram_data]
#Xs = [np.array([np.array([x, y]) for x, y in zip(item[0][:-1], item[1])]) for item in histogram_data]
for idx, X in enumerate(Xs):
    print(list(mmseqs_tables_hier.keys())[idx])
    gs = GridSearchCV(GaussianMixture(), param_grid=param_grid, scoring=bic_scorer)
    gs.fit(X)
    print(gs.best_score_, gs.best_params_)

Andryala_integrifolia_ERR7618428
-24.910008832517583 {'covariance_type': 'diag', 'n_components': 1}
Cichorium_intybus_ERR5033750
-23.818505410858307 {'covariance_type': 'diag', 'n_components': 1}
Echinops_strigosus_ERR5033325
-10.717710171342965 {'covariance_type': 'diag', 'n_components': 1}
Lactuca_floridana_ERR5033752
-28.470916426158276 {'covariance_type': 'diag', 'n_components': 1}
Pilosella_piloselloides_ERR7618430
-19.724936902708713 {'covariance_type': 'diag', 'n_components': 1}


In [178]:
mmseqs_dir = Path("/home/yjkbertrand/Documents/projects/nextpyper_simulation/miniprot_divergence/more_divergence/my50_cov100")
mmseqs_tables_hier = {file.stem:pl.read_csv(file, separator="\t") for file in sorted(mmseqs_dir.glob("*Deca*.log"))}
from plotly.subplots import make_subplots
import plotly.graph_objects as go
dfs = [mmseqs_tables_hier[key].filter(pl.col('cov') > 0.1).with_columns(pl.lit(key).alias("acc")).select("idt") for key in mmseqs_tables_hier.keys() ]
total_rows = len(dfs)
total_cols = 1
fig = make_subplots(rows=total_rows, cols=total_cols, shared_xaxes=True, vertical_spacing=0.02,subplot_titles=list(mmseqs_tables_hier.keys()),
                    shared_yaxes=True)

fig.update_layout(height=2000, width=1500, title_text=f"Similarity histograms",
    template="plotly")
for i, df in enumerate(dfs):
    fig.add_trace(go.Histogram(x=df["idt"],nbinsx=50),row=i+1,col=1)

fig.show()

In [180]:
import numpy as np
import plotly.express as px
import polars as pl
from polars import DataFrame
from scipy.optimize import curve_fit

def gaussian(x, mu, sigma, a):
    """Defines a classical Gaussian distribution."""
    return a*np.exp(-(x-mu)**2/2/sigma**2)

def get_histogram_data(df:DataFrame, mmseqs_idt_threshold:float) ->tuple[list, list]:
    """"
    Automatically infer the best number of bins in the histogram.
    Return x, y coordinates with y the height of the bar and y the central value of the bar.
    """
    nbins = len(np.histogram_bin_edges(df, bins='auto', range=(mmseqs_idt_threshold, 1)))
    tmp_fig = px.histogram(df, x="idt", nbins=nbins)
    f = tmp_fig.full_figure_for_development(warn=False)
    xbins = f.data[0].xbins
    plotbins = list(np.arange(start=xbins['start'], stop=xbins['end']+xbins['size'], step=xbins['size']))
    counts, bins = np.histogram(df, bins=plotbins)
    return (bins, counts)

def find_sim_threshold_per_acc(mmseqs_log:Path, mmseqs_idt_threshold:float=0.6, min_cov:float=0.5, flattening_value:float=0.1)->float:
    """
    Given a similarity log computed by mmseqs2, fit a Gaussian distribution to the distribution of similarity values (histogram).
    Compute the mean and standard deviation of the Gaussian distribution.

    :param mmseqs_log: the path to the mmseqs2 log file.
    :param mmseqs_idt_threshold: the mmseqs2 similarity threshold.
    :param min_cov: the minimum scaffold-probe coverage.
    :param flattening_value: bins that are lower than the largest bin multiplied by the flattening_value are removed before the Gaussian curve fitting.
    :return: the lower value at 2 standard deviations from the mean that should encompass 95% of the data.
    """
    mmseqs_tables = pl.read_csv(mmseqs_log, separator="\t")
    df = mmseqs_tables.filter(pl.col('cov') > min_cov).select("idt")
    bins, counts = get_histogram_data(df, mmseqs_idt_threshold)
    min_count = max(counts) * flattening_value
    filtered_bins, filtered_counts = zip(* [(x, y) for x, y in zip(bins[:-1], counts) if y>=min_count])
    gau_params, gau_covariance = curve_fit(gaussian, filtered_bins, filtered_counts)
    mu, sigma, _ = gau_params
    print(f"Gaussian fit: {mu}, {sigma}")
    return mu - 2*abs(sigma)

print(find_sim_threshold_per_acc(Path('/home/yjkbertrand/Documents/projects/nextpyper_simulation/miniprot_divergence/data/batrachium/mmseqs_logs/AT_01_1.log')))

Gaussian fit: 0.9450421961031129, -0.027372721716440625
0.8902967526702317


Now we need to make a parser for the miniprot log files

In [12]:
miniprot_pat = r"cds\s\[Cds\(self\.mRNA_start=(?P<qstart>\d+),self\.mRNA_end=(?P<qend>\d+),self\.probe_start=(?P<tstart>\d+),self\.probe_end=(?P<tend>\d+),self\.global_identity=(?P<idt>[\d\.]+),strand=(?P<strand>-?1),\scontig=(?P<query>.*?), global_score=(?P<score>\d+)\)\]"
pat = re.compile(miniprot_pat)

In [13]:
@dataclass
class miniprotHit:
    qstart: int
    qend: int
    tstart: int
    tend: int
    idt: float
    strand: int
    query: str
    score: int
    probe: str = field(init=False)

    @classmethod
    def from_line(cls, line):
        match = pat.search(line)
        if match:
            return cls(**match.groupdict())
        else:
            raise ValueError("Line does not match expected format")
        
    def __post_init__(self):
        self.qstart = int(self.qstart)
        self.qend = int(self.qend)
        self.tstart = int(self.tstart)
        self.tend = int(self.tend)
        self.idt = float(self.idt)
        self.strand = int(self.strand)
        self.score = int(self.score)
        self.probe = re.search(r"-(.*?)_", self.query).group(1)


log = miniprot_dir / "1766.log"
with open(log) as file:
    hits = [miniprotHit.from_line(line) for line in file if line.startswith("cds") and len(line) >= 20]

In [14]:
from collections import defaultdict
miniprot_hits = defaultdict(list)
for logfile in sorted(miniprot_dir.glob("*.log")):
    with open(logfile) as file:
        for line in file:
            if line.startswith("cds") and len(line) >= 20:
                miniprot_hits[logfile.stem].append(miniprotHit.from_line(line))

In [15]:
17*6

102

In [16]:
miniprot_hits["1766"]

[miniprotHit(qstart=286, qend=2939, tstart=0, tend=430, idt=0.9206, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1739_length_3329_cov_112.0481:2:1:373008', score=2074, probe='1766'),
 miniprotHit(qstart=317, qend=2972, tstart=0, tend=430, idt=0.9274, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1739_length_3225_cov_130.5957:1:1:421171', score=2069, probe='1766'),
 miniprotHit(qstart=317, qend=1601, tstart=0, tend=280, idt=0.7936, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1738_length_1978_cov_120.5905:1:2:238528', score=1052, probe='1766'),
 miniprotHit(qstart=296, qend=611, tstart=430, tend=518, idt=0.764, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1423_length_975_cov_33.4544:1:1:32618', score=296, probe='1766'),
 miniprotHit(qstart=291, qend=642, tstart=430, tend=518, idt=0.7525, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_2057_length_853_cov_15.2673:1:1:13023', score=378, probe='1766'),
 miniprotHit(qstart=128, qend=2779, tstart=0, tend=430, idt=0.9206, strand=1, query='AT_02_1|AT_02_1-1

In [17]:
hits

[miniprotHit(qstart=286, qend=2939, tstart=0, tend=430, idt=0.9206, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1739_length_3329_cov_112.0481:2:1:373008', score=2074, probe='1766'),
 miniprotHit(qstart=317, qend=2972, tstart=0, tend=430, idt=0.9274, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1739_length_3225_cov_130.5957:1:1:421171', score=2069, probe='1766'),
 miniprotHit(qstart=317, qend=1601, tstart=0, tend=280, idt=0.7936, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1738_length_1978_cov_120.5905:1:2:238528', score=1052, probe='1766'),
 miniprotHit(qstart=296, qend=611, tstart=430, tend=518, idt=0.764, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_1423_length_975_cov_33.4544:1:1:32618', score=296, probe='1766'),
 miniprotHit(qstart=291, qend=642, tstart=430, tend=518, idt=0.7525, strand=1, query='AT_01_1|AT_01_1-1766_EDGE_2057_length_853_cov_15.2673:1:1:13023', score=378, probe='1766'),
 miniprotHit(qstart=128, qend=2779, tstart=0, tend=430, idt=0.9206, strand=1, query='AT_02_1|AT_02_1-1